In [22]:
from PIL import Image
import os

path_lion = "./lion_tiger_dataset/Lion"
path_tiger = "./lion_tiger_dataset/tiger"

vectors = []
labels = []

def load_images():
    for i in range(1, 231):
        num = f"{i:03d}"
        filename = f"lion_{num}.jpg"
        path = os.path.join(path_lion, filename)
        img = Image.open(path).convert("L")
        # img = img.resize((112, 112))
        pixels = list(img.get_flattened_data())
        vec = [p / 255 for p in pixels]
        vectors.append(vec)
        labels.append(1)
    for i in range(1, 231):
        num = f"{i:03d}"
        filename = f"tiger_{num}.jpg"
        path = os.path.join(path_tiger, filename)
        img = Image.open(path).convert("L")
        # img = img.resize((112, 112))
        pixels = list(img.get_flattened_data())
        vec = [p / 255 for p in pixels]
        vectors.append(vec)
        labels.append(-1)
        
load_images()

In [23]:
print(len(vectors))
print(len(vectors[0]))

460
50176


In [24]:
import numpy as np

def train_test_split():
    np_vectors = np.array(vectors)
    np_labels = np.array(labels)
    n = len(np_vectors)
    shuffle_index = np.arange(n)
    np.random.shuffle(shuffle_index)
    split = int(n*0.8)
    train_index = shuffle_index[:split]
    test_index = shuffle_index[split:]
    vectors_train = np_vectors[train_index].tolist()
    vectors_test = np_vectors[test_index].tolist()
    labels_train = np_labels[train_index].tolist()
    labels_test = np_labels[test_index].tolist()
    return vectors_train, vectors_test, labels_train, labels_test

In [25]:
# import random

# def select_j(i, m):
# 	j = i
# 	while (j == i):
#         # randint是闭区间[0, m-1]
# 		j = random.randint(0, m-1)
# 	return j

def clip_alpha(a, L, H):
    if a > H:
        return H
    if a < L:
        return L
    return a

def smo(maxIter, toler, C):
    np_vectors = np.array(vectors_train)
    np_labels = np.array(labels_train)
    
    b = 0
    m, n = np_vectors.shape
    iter_num = 0
    
    alphas = np.zeros(m)
    K = np_vectors @ np_vectors.T
    E_cache = np.zeros(m)

    total_iter = 0
    count_change_small = 0
    count_LH = 0
    count_eta = 0
    
    while(iter_num < maxIter):
        total_iter += 1
        if total_iter % 10 == 0:
            print(f"总迭代次数:{total_iter}")
            # print(f"alpha[j]变化太小:{count_change_small}")
            # print(f"L==H:{count_LH}")
            # print(f"eta>=0:{count_eta}")
            w = get_w(alphas)
            train_accuracy, test_accuracy = calc_accuracy(w, b)
            print(f"训练集准确率:{train_accuracy}")
            print(f"测试集准确率:{test_accuracy}")
            
        alphaPairsChanged = 0
        for i in range(m):
            fxi = np.sum(alphas*np_labels*K[:, i]) + b
            Ei = fxi - np_labels[i]
            E_cache[i] = Ei
                
            if ((np_labels[i]*Ei < -toler) and (alphas[i] < C)) or ((np_labels[i]*Ei > toler) and (alphas[i] > 0)):
                # j = select_j(i, m)
                # j = np.argmax(np.abs(E_cache - Ei))
                candidates = np.argsort(np.abs(E_cache - Ei))[::-1]
                for j in candidates[:5]:
                    fxj = np.sum(alphas*np_labels*K[:, j]) + b
                    Ej = fxj - np_labels[j]    
                    alpha_i_old = alphas[i].copy()
                    alpha_j_old = alphas[j].copy()
                    if np_labels[i] != np_labels[j]:
                        L = max(0, alphas[j] - alphas[i])
                        H = min(C, C + alphas[j] - alphas[i])
                    else:
                        L = max(0, alphas[j] + alphas[i] - C)
                        H = min(C, alphas[j] + alphas[i])
                    # L == H时, alpha_j_new == alpha_j_old == L == H, 没有调整空间
                    if L == H:
                        # print("L == H")
                        count_LH += 1
                        continue
                    eta = 2 * K[i, j] - K[i, i] - K[j, j]
                    # 要最大化W函数(得到最优alphas), 其二次项系数为eta/2, 当eta < 0时, 求导可得到极大值点, 再裁剪到L, H区间
                    # 线性核eta <= 0, 这里主要是排除eta == 0的可能
                    if eta >= 0:
                        # print("eta >= 0")
                        # if eta == 0:
                        #     print("eta == 0")
                        count_eta += 1
                        continue
                    #这里是对W求导等于0后解出的参数alpha的值后, 更新原来的值
                    alphas[j] -= np_labels[j]*(Ei - Ej)/eta
                    alphas[j] = clip_alpha(alphas[j], L, H)
                    if (abs(alphas[j] - alpha_j_old) < 0.00001):
                        # print("alpha[j]变化太小")
                        count_change_small += 1
                        continue
                    alphas[i] += np_labels[i]*np_labels[j]*(alpha_j_old - alphas[j])
                    b1 = b - Ei - np_labels[i]*(alphas[i] - alpha_i_old)*K[i, i] - np_labels[j]*(alphas[j] - alpha_j_old)*K[i, j]
                    b2 = b - Ej - np_labels[i]*(alphas[i] - alpha_i_old)*K[i, j] - np_labels[j]*(alphas[j] - alpha_j_old)*K[j, j]
                    if (alphas[i] > 0) and (alphas[i] < C):
                        b = b1
                    elif (alphas[j] > 0) and (alphas[j] < C):
                        b = b2
                    else:
                        b = (b1 + b2)/2
                    alphaPairsChanged += 1
                    break
                # print(f"第{iter_num}次迭代, 样本:{i}, alpha优化次数:{alphaPairsChanged}")
        if alphaPairsChanged == 0:
            iter_num += 1
        else:
            iter_num = 0
        # print(f"迭代次数:{iter_num}")
    return alphas, b

In [26]:
def get_w(alphas):
    np_alphas = np.array(alphas)
    np_vectors = np.array(vectors_train)
    np_labels = np.array(labels_train)
    # w = sum(alpha_i * label_i * vector_i)
    w = np.dot(np_vectors.T, np_alphas * np_labels)
    return w.tolist()

In [27]:
def predict(x, w, b):
    score = np.dot(w, x) + b
    return 1 if score >= 0 else -1

def calc_accuracy(w, b):
    correct = 0
    for vec, label in zip(vectors_train, labels_train):
        if predict(vec, w, b) == label:
            correct += 1
    train_accuracy = correct / len(vectors_train)
    correct = 0
    for vec, label in zip(vectors_test, labels_test):
        if predict(vec, w, b) == label:
            correct += 1
    test_accuracy = correct / len(vectors_test)
    return train_accuracy, test_accuracy

In [28]:
train_avg_accuracy, test_avg_accuracy = 0, 0
for i in range(10):
    print(f"第{i}轮开始")
    vectors_train, vectors_test, labels_train, labels_test = train_test_split()
    alphas, b = smo(40, 0.01, 0.6)
    w = get_w(alphas)
    train_accuracy, test_accuracy = calc_accuracy(w, b)
    train_avg_accuracy += train_accuracy
    test_avg_accuracy += test_accuracy
    print(f"训练集准确率:{train_accuracy}")
    print(f"测试集准确率:{test_accuracy}")
train_avg_accuracy /= 10
test_avg_accuracy /= 10
print(f"训练集平均准确率:{train_avg_accuracy}")
print(f"测试集平均准确率:{test_avg_accuracy}")

第0轮开始
总迭代次数:10
训练集准确率:0.8913043478260869
测试集准确率:0.6195652173913043
总迭代次数:20
训练集准确率:0.9891304347826086
测试集准确率:0.6521739130434783
总迭代次数:30
训练集准确率:0.9945652173913043
测试集准确率:0.6304347826086957
总迭代次数:40
训练集准确率:1.0
测试集准确率:0.6739130434782609
总迭代次数:50
训练集准确率:1.0
测试集准确率:0.6956521739130435
总迭代次数:60
训练集准确率:1.0
测试集准确率:0.6739130434782609
总迭代次数:70
训练集准确率:1.0
测试集准确率:0.6739130434782609
总迭代次数:80
训练集准确率:1.0
测试集准确率:0.6739130434782609
总迭代次数:90
训练集准确率:1.0
测试集准确率:0.6739130434782609
训练集准确率:1.0
测试集准确率:0.6739130434782609
第1轮开始
总迭代次数:10
训练集准确率:0.875
测试集准确率:0.5760869565217391
总迭代次数:20
训练集准确率:0.9891304347826086
测试集准确率:0.6630434782608695
总迭代次数:30
训练集准确率:0.9891304347826086
测试集准确率:0.6413043478260869
总迭代次数:40
训练集准确率:1.0
测试集准确率:0.6630434782608695
总迭代次数:50
训练集准确率:1.0
测试集准确率:0.6521739130434783
总迭代次数:60
训练集准确率:1.0
测试集准确率:0.6521739130434783
总迭代次数:70
训练集准确率:1.0
测试集准确率:0.6521739130434783
总迭代次数:80
训练集准确率:1.0
测试集准确率:0.6521739130434783
训练集准确率:1.0
测试集准确率:0.6521739130434783
第2轮开始
总迭代次数:10
训练集准确率:0.9592391304347826
测试集准确率:0.66304